In [ ]:
import pandas as pd
import csv
import pickle
from sklearn.linear_model import Lasso
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
from pathlib import Path
import os
import random
from itertools import product


from stage1 import lasso_rolling_window, calculate_r_squared
from stage2 import estimate_kappa_curve_fit, compute_alm_returns, compute_stage2_r_squared
from grid_search import grid_search

In [ ]:
# load feature matrix and response variable
feature_matrix = pd.read_csv("../../data/merged_return_topic_data.csv", index_col=0, parse_dates=True)
response_variables = pd.read_csv("../../data/response.csv", index_col=0, parse_dates=True)

In [ ]:
# select a response variable (either marekt return or sp500 return)
y = response_variables['sprtrn']  # or 'sprtrn' for SP500 returns or vwretx
# transform returns to log
y = np.log(y+1)

# create feature matrix
X = feature_matrix.copy()
random.seed(42)

# Separate topic (name) and stock (numeric) columns
topic_cols = [col for col in X.columns if not str(col).isdigit()]
stock_cols = [col for col in X.columns if str(col).isdigit()]

# User-selected number of each (or use len(...) for "all")
num_topics = len(topic_cols)  # e.g., 100
num_stocks = 0

# Randomly sample (without replacement), limited by available count
selected_topics = random.sample(topic_cols, min(num_topics, len(topic_cols)))
selected_stocks = random.sample(stock_cols, min(num_stocks, len(stock_cols)))

# Final filtered dataframe
X = X[selected_topics + selected_stocks]

# Convert stock returns to log returns: log(1+r)
X[selected_stocks] = np.log(X[selected_stocks] + 1)

# ensure that all indices align
common_index = X.index.intersection(y.index)
X = X.loc[common_index]
y = y.loc[common_index]

In [ ]:
# Define your grid
lambda_values = [ 0.002723 * factor for factor in [0.6, 0.8, 1, 1.2, 1.4]]
param_grid = {
    'window_sizes': [275,300,325],
    'n_lags': [5,6,7],
    'lambdas': lambda_values
}

# Run grid search 
summary_df, coefficients_df = grid_search(X, y, param_grid, verbose=True)

In [ ]:
# Run a loop to estimate grid, find best parameters, estimate grid around best parameters, repeat

# Objective Function
def objective_function(row):
    if 1.96 < row['kappa_tstat'] <= 20:
        return row['r2_insample_stage2'] * row['kappa']
    else:
        return -float('inf')


# number of loops
refinement_rounds = 3

# Initial grid 
window_sizes = [50,100,200,300,400,500]
n_lags = [1,2,3,4,5,6,7,8,9,10]
lambda_base = 0.003
lambda_values = [lambda_base * factor for factor in [0.8, 0.9, 1.0, 1.1, 1.2]]

# IMPORTANT: match grid_search expectations: window_sizes, n_lags, lambdas
current_param_grid = {
    'window_sizes': window_sizes,
    'n_lags': n_lags,
    'lambdas': lambda_values
}

results_all_rounds = []

# ----------------------------------------
# Loop
# ----------------------------------------
for i in range(refinement_rounds):

    print(f"\n🔷 Starting Refinement Round {i+1}")

    # Run grid search
    summary_df, coefficients_df = grid_search(
        X, y, current_param_grid, verbose=True
    )

    # Evaluate objective
    summary_df['objective'] = summary_df.apply(objective_function, axis=1)

    # Track results
    results_all_rounds.append(summary_df.copy())

    # Pick best point
    best_row = summary_df.loc[summary_df['objective'].idxmax()]
    best_window = int(best_row['window_size'])
    best_n_lags = int(best_row['n_lags'])
    best_lambda = float(best_row['lambda'])

    print(f"Best so far (round {i+1}):")
    print(best_row[['window_size', 'n_lags', 'lambda', 'objective']])

    # ----------------------------------------
    # build new grid around best point
    # ----------------------------------------

    # shrink window search range
    window_sizes_refined = list(range(best_window - 25, best_window + 26, 5))
    window_sizes_refined = [w for w in window_sizes_refined if w > 20]

    # shrink n_lags search range
    n_lags_refined = list(range(best_n_lags - 1, best_n_lags + 2))
    n_lags_refined = [l for l in n_lags_refined if l >= 1]

    # shrink lambda range (currently ±20% around best)
    lambda_values_refined = np.linspace(
        0.8 * best_lambda,
        1.2 * best_lambda,
        num=10
    )

    # update grid for next iteration
    # AGAIN: keep keys consistent with grid_search: window_sizes, n_lags, lambdas
    current_param_grid = {
        'window_sizes': window_sizes_refined,
        'n_lags': n_lags_refined,
        'lambdas': lambda_values_refined
    }


# -------------------------------------------------
# 🏆 Final Result
# -------------------------------------------------
final_df = pd.concat(results_all_rounds, ignore_index=True)
final_df['objective'] = final_df.apply(objective_function, axis=1)
best_overall = final_df.loc[final_df['objective'].idxmax()]

print("\n🏆 Best Hyperparameters After Refinement:")
print(best_overall[['window_size', 'n_lags', 'lambda', 'objective']])

print("\n🏆 Best Hyperparameters After Refinement:")
print(best_overall[['window_size','n_lags','lambda','objective']])


In [18]:
# print for the 5 rows with the highest objective values r2 insample_stage2 and stage1, kappa ,kappa_tstat, lambda, window_size, n_lags
top_10 = final_df.nlargest(10, 'objective')
print(top_10[['r2_insample_stage2', 'r2_insample_stage1', 'kappa', 'kappa_tstat', 'lambda', 'window_size', 'n_lags']])

     r2_insample_stage2  r2_insample_stage1     kappa  kappa_tstat    lambda  \
623            0.001584           -0.000903  0.896122     9.150980  0.002295   
230            0.001150           -0.000987  0.949137    18.712327  0.002400   
716            0.001141           -0.000987  0.948164    18.354210  0.002399   
432            0.001202           -0.000940  0.894788     8.146114  0.002347   
302            0.001126           -0.000953  0.837186     5.558609  0.002453   
299            0.000987           -0.000882  0.891301     9.230826  0.002700   
612            0.000881           -0.000916  0.889099     6.860909  0.002295   
738            0.000859           -0.000833  0.840627     7.923127  0.002399   
307            0.000863           -0.000825  0.832507     5.090370  0.002240   
700            0.000776           -0.001023  0.887435     8.947722  0.002503   

     window_size  n_lags  
623          305       1  
230          300       1  
716          300       1  
432        

In [19]:
final_df2 = pd.read_parquet('grid_search_optimization.parquet')

In [ ]:
# save
final_df.to_parquet('grid_search_optimization.parquet', index=False)

In [ ]:
summary_df.to_parquet('grid_search_summary_9.parquet', index=False)


In [ ]:
coefficients_df.to_parquet('grid_search_coefficients_9.parquet', index=False)